<a href="https://colab.research.google.com/github/mvashi-sonic/AICapstoneProj/blob/dev/Evaluate_FAISS_RERANKER.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
pip install sentence-transformers faiss-cpu

In [2]:
from google.colab import drive

# 1. Mount your Google Drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import json
import pickle
import numpy as np
import faiss

from sentence_transformers import SentenceTransformer

In [4]:
from transformers import AutoTokenizer
from transformers import AutoModelForSequenceClassification
import sys
import transformers
import torch

In [ ]:
embedder = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)
print(embedder.get_embedding_dimension())

In [6]:
chunks = []
dict = {}
test_rec = open("test_records.jsonl", "a")#with open("/content/drive/MyDrive/capstone_usable_qa_data/all_records.json", encoding="utf-8") as f:
with open("/content/drive/MyDrive/AI_capstone_training_data/positive_records_singular.json", encoding="utf-8") as f:
    records = json.load(f)
    #for record in records["All_Records"]:
    for record in records["Positive_Records"]:
      if(record["chunk_id"] not in dict):
        dict[record["chunk_id"]] = 1
        chunks.append(record)

    print(f"Distinct Records: {len(chunks)}")
json.dump(chunks,test_rec )

Distinct Records: 20691


In [7]:
index = faiss.read_index(
    "/content/drive/MyDrive/capstone_FAISS_embeddings/financial_reports.index"
)
with open("/content/drive/MyDrive/capstone_FAISS_embeddings/financial_reports_metadata.pkl","rb") as f:
    metadata = pickle.load(f)

In [8]:
from transformers import AutoTokenizer
from transformers import AutoModelForSequenceClassification


tokenizer = AutoTokenizer.from_pretrained(
    "/content/drive/MyDrive/AI_Capstone_Model_Final/Best_Working"
)

model = AutoModelForSequenceClassification.from_pretrained(
    "/content/drive/MyDrive/AI_Capstone_Model_Final/Best_Working"
)


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [9]:
device = torch.device("cpu")

In [10]:
import torch
import numpy as np


def rerank_candidates(
    question,
    candidate_indices,
    metadata,
    tokenizer,
    model,
    device,
    batch_size=32
):
    """
    Rerank FAISS candidates using FinBERT relevance probability.
    """

    candidates = [
        metadata[idx]
        for idx in candidate_indices
        if idx != -1
    ]

    reranked = []

    model.eval()

    for start in range(0, len(candidates), batch_size):

        batch_candidates = candidates[
            start:start + batch_size
        ]

        paragraphs = [
            c["reference"]
            for c in batch_candidates
        ]

        questions = [
            question
        ] * len(paragraphs)

        inputs = tokenizer(
            questions,
            paragraphs,
            padding=True,
            truncation=True,
            max_length=512,
            return_tensors="pt"
        )

        inputs = {
            k: v.to(device)
            for k, v in inputs.items()
        }

        with torch.no_grad():

            logits = model(**inputs).logits

            probs = torch.softmax(
                logits,
                dim=1
            )[:, 1]

        probs = probs.cpu().numpy()

        for candidate, probability in zip(
            batch_candidates,
            probs
        ):

            reranked.append({
                "chunk_id": candidate["chunk_id"],
                "reference": candidate["reference"],
                "finbert_score": float(probability)
            })

    reranked.sort(
        key=lambda x: x["finbert_score"],
        reverse=True
    )

    return reranked

In [12]:
def evaluate_faiss_and_reranker(
    eval_records,
    embedder,
    index,
    metadata,
    tokenizer,
    model,
    device,
    faiss_k=50,
    k_values=(1, 5, 10, 20),
    rerank_batch_size=32
):


    # Metric containers
    print("Metric containers")
    faiss_hits = {
        k: 0
        for k in k_values
    }

    reranker_hits = {
        k: 0
        for k in k_values
    }

    faiss_reciprocal_ranks = []
    reranker_reciprocal_ranks = []

    total_questions = len(eval_records)

    # Evaluate each question
    print("Evaluate each question")
    for i, record in enumerate(eval_records):
        print(f"{i}/{total_questions}")
        # Extract question and correct chunk ID
        print("Extract question and correct chunk ID")
        question = record["question"]
        correct_chunk_id = record["chunk_id"]

        # 1. MiniLM question embedding
        query_embedding = embedder.encode(
            [question],
            convert_to_numpy=True,
            normalize_embeddings=True
        )

        query_embedding = np.asarray(
            query_embedding,
            dtype=np.float32
        )

        # 2. FAISS retrieval
        faiss_scores, faiss_ids = index.search(
            query_embedding,
            faiss_k
        )

        candidate_indices = faiss_ids[0]

        faiss_chunk_ids = [
            metadata[idx]["chunk_id"]
            for idx in candidate_indices
            if idx != -1
        ]

        # 3. FAISS Recall@K

        for k in k_values:

            if correct_chunk_id in faiss_chunk_ids[:k]:

                faiss_hits[k] += 1

        # 4. FAISS MRR
        if correct_chunk_id in faiss_chunk_ids:

            rank = (
                faiss_chunk_ids.index(
                    correct_chunk_id
                ) + 1
            )

            faiss_reciprocal_ranks.append(
                1.0 / rank
            )

        else:

            faiss_reciprocal_ranks.append(
                0.0
            )

        # 5. FinBERT reranking
        print("FinBERT reranking")
        reranked = rerank_candidates(
            question=question,
            candidate_indices=candidate_indices,
            metadata=metadata,
            tokenizer=tokenizer,
            model=model,
            device=device,
            batch_size=rerank_batch_size
        )

        reranked_chunk_ids = [
            result["chunk_id"]
            for result in reranked
        ]
        # 6. Reranker Recall@K
        print("Reranker Recall@K")
        for k in k_values:

            if correct_chunk_id in reranked_chunk_ids[:k]:

                reranker_hits[k] += 1

        # 7. Reranker MRR
        print("Reranker MRR")
        if correct_chunk_id in reranked_chunk_ids:

            rank = (
                reranked_chunk_ids.index(
                    correct_chunk_id
                ) + 1
            )

            reranker_reciprocal_ranks.append(
                1.0 / rank
            )

        else:

            reranker_reciprocal_ranks.append(
                0.0
            )

        # Progress display
        if (i + 1) % 100 == 0:

            print(
                f"Processed "
                f"{i + 1}/{total_questions}"
            )

    # Calculate final metrics
    print("Calculate final metrics")
    results = {
        "FAISS": {},
        "FAISS + FinBERT": {}
    }

    for k in k_values:

        results["FAISS"][f"Recall@{k}"] = (
            faiss_hits[k]
            / total_questions
        )

        results[
            "FAISS + FinBERT"
        ][f"Recall@{k}"] = (
            reranker_hits[k]
            / total_questions
        )

    results["FAISS"]["MRR"] = (
        np.mean(
            faiss_reciprocal_ranks
        )
    )

    results["FAISS + FinBERT"]["MRR"] = (
        np.mean(
            reranker_reciprocal_ranks
        )
    )
    print("Done")
    return results

In [13]:
results = evaluate_faiss_and_reranker(
    eval_records=chunks[:1000],
    embedder=embedder,
    index=index,
    metadata=metadata,
    tokenizer=tokenizer,
    model=model,
    device=device,
    faiss_k=50,
    k_values=(5, 10),
    rerank_batch_size=32
)

Streaming output truncated to the last 5000 lines.
FinBERT reranking
Reranker Recall@K
Reranker MRR
3/1000
Extract question and correct chunk ID
FinBERT reranking
Reranker Recall@K
Reranker MRR
4/1000
Extract question and correct chunk ID
FinBERT reranking
Reranker Recall@K
Reranker MRR
5/1000
Extract question and correct chunk ID
FinBERT reranking
Reranker Recall@K
Reranker MRR
6/1000
Extract question and correct chunk ID
FinBERT reranking
Reranker Recall@K
Reranker MRR
7/1000
Extract question and correct chunk ID
FinBERT reranking
Reranker Recall@K
Reranker MRR
8/1000
Extract question and correct chunk ID
FinBERT reranking
Reranker Recall@K
Reranker MRR
9/1000
Extract question and correct chunk ID
FinBERT reranking
Reranker Recall@K
Reranker MRR
10/1000
Extract question and correct chunk ID
FinBERT reranking
Reranker Recall@K
Reranker MRR
11/1000
Extract question and correct chunk ID
FinBERT reranking
Reranker Recall@K
Reranker MRR
12/1000
Extract question and correct chunk ID
FinBER

In [14]:
for system, metrics in results.items():

    print("\n", system)
    print("-" * 40)

    for metric, value in metrics.items():

        print(
            f"{metric:<12} {value:.4f}"
        )


 FAISS
----------------------------------------
Recall@5     0.6410
Recall@10    0.7270
MRR          0.5030

 FAISS + FinBERT
----------------------------------------
Recall@5     0.8210
Recall@10    0.8630
MRR          0.6310


In [ ]:
model.eval()